# Cleaning of the logistics data

A generative AI (Claude) was used to review and correct this cleaning and to add comments.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Data exploration

In [2]:
data_dictionary = pd.read_csv("../data/data_dictionary.csv")
data_dictionary

,source_bundle,source_system_id,file_name,field_name,logical_type,key_role,description,example_value
0,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_number,identifier,primary_or_composite_key,Source-local encounter identifier.,ENC-000001
1,SB02,SYS-CLN-OPS,clinical_encounters.csv,patient_number,identifier,non_key,Source-local patient identifier used within th...,PAT-31162
2,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_start,date_or_datetime,non_key,Encounter start.,2018-02-27T09:15:00
3,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_end,date_or_datetime,non_key,Encounter end.,2018-02-27T10:45:00
4,SB02,SYS-CLN-OPS,clinical_encounters.csv,facility_license_number,identifier,non_key,Facility license number.,FAC-VLN-101
...,...,...,...,...,...,...,...,...
350,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,named_location,categorical_or_text,non_key,Named location.,Valenne Central District
351,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,title,categorical_or_text,non_key,Title.,Restricted Review of Recurrent Executive Healt...
352,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,text,long_text,non_key,Full document body as stored by the source rep...,PURPOSE\r\nThis restricted briefing summarizes...
353,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,public_or_restricted,categorical_or_text,non_key,Public or restricted.,restricted


In [3]:
domains = ["public_records", "health", "ecology", "logistics"]
dataframes = {}

for domain in domains:
    data_dir = Path(f"../data/{domain}")
    csv_files = list(data_dir.glob("*.csv"))
    for file in csv_files:
        name = file.stem
        dataframes[name] = pd.read_csv(file)
        print(f"--- {domain} / {name} ---")
        print(dataframes[name].shape)
        print(dataframes[name].dtypes)
        print(dataframes[name].isna().sum())
        print()

--- public_records / institutional_memos ---
(30, 12)
document_id                str
document_date              str
document_type              str
author_name                str
author_organization        str
recipient_text             str
subject_line               str
text                       str
related_purchase_order     str
related_shipment_number    str
related_case_number        str
visibility_level           str
dtype: object
document_id                 0
document_date               0
document_type               0
author_name                 0
author_organization         0
recipient_text              0
subject_line                0
text                        0
related_purchase_order     25
related_shipment_number    22
related_case_number        20
visibility_level            0
dtype: int64

--- public_records / licensed_facilities ---
(3, 12)
facility_license_number                    str
facility_name                              str
facility_category                      

### Helper functions

In [4]:
# Date formats found in the raw files.
# Dates with "/" and the year at the end are in day/month/year format:
# "30/06/2025" (goods_receipts) or "16/02/2018 12:00:00" (shipments) cannot be read otherwise.
DATE_FORMATS = [
    "%Y-%m-%d", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
    "%Y/%m/%d", "%Y/%m/%d %H:%M:%S",
    "%d/%m/%Y", "%d/%m/%Y %H:%M:%S",
    "%d-%b-%Y", "%d-%b-%Y %H:%M:%S",
]

def parse_dates_robust(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series  # already a date

    # Convert everything to clean text, avoiding the "nan" or "NaT" strings
    s = series.astype("string").str.strip()
    s = s.replace(["nan", "NaT", "None", ""], pd.NA)

    # Try each format one by one (with format="mixed", "12/07/2024" would be read as 7 December)
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in DATE_FORMATS:
        to_parse = result.isna() & s.notna()
        if not to_parse.any():
            break
        result[to_parse] = pd.to_datetime(s[to_parse], format=fmt, errors="coerce")

    unparsed = s.notna() & result.isna()
    if unparsed.any():
        print(f"Warning: {unparsed.sum()} unrecognised date(s): {s[unparsed].unique()[:5].tolist()}")
    return result

In [5]:
def non_numeric_values(df, column):
    non_num = pd.to_numeric(df[column], errors="coerce").isna() & df[column].notna()
    return df.loc[non_num, [column]].value_counts()

In [6]:
def str_to_num(table_name, column):
    s = dataframes[table_name][column].astype(str)
    # Extract the first number in the cell, keeping the "-" sign:
    # "-2.9 case" -> -2.9 ; "56,41" -> 56.41 ; "14 kit" -> 14
    s_clean = s.str.extract(r'(-?\d+(?:[.,]\d+)?)', expand=False).str.replace(',', '.', regex=False)
    dataframes[table_name][column] = pd.to_numeric(s_clean, errors='coerce').astype(float)

In [7]:
def quick_review(df, csv_name, dictionary):
    print(f"====== Full review of {csv_name} ======")
    print(f"Size: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Duplicate rows: {df.duplicated().sum()}\n")

    # Overall summary of missing values
    gaps = df.isna().sum()
    columns_with_gaps = gaps[gaps > 0]
    if len(columns_with_gaps) > 0:
        print("[Gaps found] Missing values (NaN) per column:")
        print(columns_with_gaps)
        print()

    dictionary_rules = dictionary[dictionary["file_name"] == csv_name]

    if len(dictionary_rules) == 0:
        print(f"ERROR: no record found for '{csv_name}' in the dictionary.\n")
        return

    print("--- Check by logical type (dictionary vs DataFrame) ---")

    for _, row in dictionary_rules.iterrows():
        col = row["field_name"]
        expected_type = row["logical_type"]
        role = row["key_role"]

        if col in df.columns:
            current_dtype = str(df[col].dtype)

            # Check for mixed types in the column
            # NaN (NoneType) are ignored so that only real values are counted
            types_present = df[col].dropna().apply(type).nunique()
            if types_present > 1:
                type_details = df[col].dropna().apply(type).value_counts().to_dict()
                print(f"[Mixed types] '{col}' contains several different types: {type_details}")

            # Check numeric columns
            if expected_type == "numeric":
                if not pd.api.types.is_numeric_dtype(df[col]):
                    print(f"[Numeric] '{col}' is read as text ({current_dtype}) -> needs cleaning!")
                else:
                    print(f"[Numeric] '{col}': number format OK.")

            # Check dates
            elif expected_type == "date_or_datetime":
                if not pd.api.types.is_datetime64_any_dtype(df[col]):
                    print(f"[Date] '{col}' is read as text ({current_dtype}) -> needs conversion!")
                else:
                    print(f"[Date] '{col}': date format OK.")

            # Check primary keys
            if role == "primary_or_composite_key":
                n_key_duplicates = df[col].duplicated().sum()
                if n_key_duplicates > 0:
                    print(f"[Primary key] '{col}' contains {n_key_duplicates} duplicates!")
                else:
                    print(f"[Primary key] '{col}': unique and valid.")
        else:
            print(f"Column '{col}' listed in the dictionary is missing from the DataFrame.")

    print("\n")
    print("==================================================\n")

In [8]:
import re

def normalize_text(df, categorical_columns):
    # 1) All text columns: strip leading/trailing spaces and collapse multiple spaces
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        df[col] = df[col].map(lambda v: re.sub(r"\s+", " ", v).strip() if isinstance(v, str) else v)

    # 2) Categorical columns: "POSTED" and "Posted" become the most frequent spelling ("posted")
    for col in categorical_columns:
        key = df[col].map(lambda v: v.lower() if isinstance(v, str) else v)
        majority_spelling = df[col].groupby(key).agg(lambda x: x.value_counts().index[0])
        df[col] = key.map(majority_spelling)
    return df

#### Cleaning of erp_business_partners

Problems: empty or whitespace-only cells, `onboarding_date` read as text, one missing billing region. Fixes: empty cells → NaN, date conversion, region deduced from the currency (each currency matches a single country).

In [9]:
quick_review(dataframes["erp_business_partners"], "erp_business_partners.csv", data_dictionary)


====== Full review of erp_business_partners.csv ======
Size: 80 rows, 11 columns
Duplicate rows: 0

[Gaps found] Missing values (NaN) per column:
external_registration_number    14
billing_region                   1
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'partner_code': unique and valid.
[Date] 'onboarding_date' is read as text (str) -> needs conversion!





In [10]:
# 1. Replace cells that contain only spaces (apparently empty) with NaN
dataframes["erp_business_partners"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Replace specific keywords with real NaN
dataframes["erp_business_partners"].replace(["N/A", "-", "Unknown", "null", ""], np.nan, inplace=True)

,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date
0,BP-30001,Marula Cold Store Cooperative,supplier_customer,REG-AVL-20012,Avelora Federation,Eastern Avelora Agricultural Cluster,AVL,NET45,active,review,2001-02-01
1,BP-30002,Blue Heron Monitoring Cooperative,customer,REG-KNT-20037,Kintara Republic,Kintara Karst Cluster,KRN,NET45,active,standard,2012-03-01
2,BP-30003,Orra Technical Services,supplier,REG-AVL-20022,Avelora Federation,Eastern Avelora Agricultural Cluster,AVL,PUBLIC30,active,review,2008-04-01
3,BP-30004,Common Ground Climate Forum,service_partner,REG-VLN-01008,Valenne Republic,Valenne Central District,VLN,PREPAID,active,review,2014-05-01
4,BP-30005,Warden Medical Ventures,service_partner,REG-VLN-01016,Valenne Republic,Valenne Republic,VLN,PREPAID,inactive,standard,2013-06-01
...,...,...,...,...,...,...,...,...,...,...,...
75,BP-30076,Avelora Council of State,customer,NaN,Avelora Federation,Avelora Federation,AVL,PUBLIC30,active,standard,1999-05-01
76,BP-30077,Velin Packaging Industries,supplier,REG-AVL-20021,Avelora Federation,Eastern Avelora Agricultural Cluster,AVL,PUBLIC30,active,standard,1998-06-01
77,BP-30078,Meridian BioLogistics,carrier,REG-VLN-01001,Valenne Republic,Valenne Central District,VLN,NET30,active,standard,2011-07-01
78,BP-30079,Kintara Field Equipment Cooperative,supplier_customer,REG-KNT-20006,Kintara Republic,Kintara Karst Cluster,KRN,NET30,active,elevated,2003-08-01


In [11]:
# 1. Date conversion (same function as for the other tables: "01/10/2003" = 1 October 2003)
dataframes["erp_business_partners"]["onboarding_date"] = parse_dates_robust(
    dataframes["erp_business_partners"]["onboarding_date"]
)

In [12]:
# 2. Fill the gap in 'billing_region'.
# 'shipping_region' is not at the same geographic level ("Kintara Karst Cluster" is a region
# inside "Kintara Republic"): using it would create a country category that does not exist.
# However, among the 79 partners with a value, each currency matches a single billing country
# (KRN -> Kintara Republic, VLN -> Valenne Republic, ...). We learn this mapping from the data
# and use it to fill the missing value (the prefix of the registration number, e.g. REG-KNT, confirms it).
bp = dataframes["erp_business_partners"]
known = bp.dropna(subset=["billing_region"])
country_by_currency = known.groupby("default_currency")["billing_region"].agg(lambda s: s.unique().tolist())
print(country_by_currency)
assert country_by_currency.map(len).eq(1).all(), "A currency matches several countries: imputation is unreliable"

missing = bp["billing_region"].isna()
bp.loc[missing, "billing_region"] = bp.loc[missing, "default_currency"].map(country_by_currency.str[0])
display(bp.loc[missing, ["partner_code", "external_registration_number", "default_currency", "shipping_region", "billing_region"]])

default_currency
AVL      [Avelora Federation]
KRN        [Kintara Republic]
NDH         [Nordhaven Union]
TMB    [Tembara Commonwealth]
VLN        [Valenne Republic]
Name: billing_region, dtype: object

,partner_code,external_registration_number,default_currency,shipping_region,billing_region
28,BP-30029,REG-KNT-20014,KRN,Kintara Karst Cluster,Kintara Republic


#### Cleaning of erp_facilities

Problems: `opening_date` read as text, one missing value in `storage_capabilities`. Fixes: date conversion; the missing value is left empty, nothing allows us to guess it.

In [13]:
quick_review(dataframes["erp_facilities"], "erp_facilities.csv", data_dictionary)

====== Full review of erp_facilities.csv ======
Size: 30 rows, 11 columns
Duplicate rows: 0

[Gaps found] Missing values (NaN) per column:
storage_capabilities    1
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'facility_code': unique and valid.
[Numeric] 'latitude': number format OK.
[Numeric] 'longitude': number format OK.
[Date] 'opening_date' is read as text (str) -> needs conversion!





In [14]:
# 1. Replace cells that contain only spaces (apparently empty) with NaN
dataframes["erp_facilities"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Replace specific keywords with real NaN
dataframes["erp_facilities"].replace(["N/A", "-", "Unknown", "null", ""], np.nan, inplace=True)

,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities
0,FCL-1011,Meridian Valenne Headquarters,office,BP-30078,"Meridian Valenne Headquarters, Valenne Central...",Valenne Central District,47.2400,7.5800,2006-01-01,active,ambient
1,FCL-1024,Meridian Kintara Biological Aggregation Facility,warehouse,BP-30078,Meridian Kintara Biological Aggregation Facili...,Kintara Karst Cluster,4.7400,112.2800,1997-01-01,active,ambient;secure_cage;2-8C;frozen
2,FCL-1019,Meridian Avelora Transfer Annex,transfer_hub,BP-30078,"Meridian Avelora Transfer Annex, Eastern Avelo...",Eastern Avelora Agricultural Cluster,7.4600,101.9100,2003-01-01,active,ambient;secure_cage;2-8C;frozen
3,FCL-1020,Meridian Tembara Coastal Hub,transfer_hub,BP-30078,"Meridian Tembara Coastal Hub, Tembara Coastal ...",Tembara Coastal Cluster,2.2000,104.5500,2010-01-01,active,ambient;secure_cage;2-8C;frozen
4,FCL-1016,Northstar Tembara Cold-Chain Hub,warehouse,BP-30047,"Northstar Tembara Cold-Chain Hub, Tembara Coas...",Tembara Coastal Cluster,2.2300,104.5800,2008-01-01,active,ambient;secure_cage;2-8C;frozen
5,FCL-1010,Asterion Central Clinical Campus,laboratory_delivery_point,BP-30021,"Asterion Central Clinical Campus, Valenne Cent...",Valenne Central District,47.2000,7.5100,2008-01-01,active,2-8C;controlled_access
6,FCL-1007,Asterion Avelora Executive Clinic,clinical_delivery_point,BP-30021,"Asterion Avelora Executive Clinic, Eastern Ave...",Eastern Avelora Agricultural Cluster,7.3500,101.6200,2011-01-01,active,2-8C;controlled_access
7,FCL-1030,Kintara Cave Authority Field Depot,field_depot,BP-30016,"Kintara Cave Authority Field Depot, Kintara Ka...",Kintara Karst Cluster,4.7600,112.3800,2011-01-01,active,ambient
8,FCL-1029,Vale Strategy Valenne Office,office,BP-30044,"Vale Strategy Valenne Office, Valenne Central ...",Valenne Central District,47.2600,7.4900,2009-01-01,active,NaN
9,FCL-1015,Threshold Earth Avelora Laboratory,laboratory_delivery_point,BP-30040,"Threshold Earth Avelora Laboratory, Eastern Av...",Eastern Avelora Agricultural Cluster,7.5000,101.7500,2012-01-01,active,2-8C;controlled_access


In [15]:
dataframes["erp_facilities"]["opening_date"] = parse_dates_robust(dataframes["erp_facilities"]["opening_date"])


#### Cleaning of goods_receipts

Problems: 2 duplicate rows (same `receipt_number`), dates in several formats (including day/month/year), received quantities stored as text. Fixes: duplicates removed, dates standardised, quantities converted to numbers.

In [16]:
quick_review(dataframes["goods_receipts"], "goods_receipts.csv", data_dictionary)

====== Full review of goods_receipts.csv ======
Size: 612 rows, 12 columns
Duplicate rows: 2

[Gaps found] Missing values (NaN) per column:
inspection_status    4
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'receipt_number' contains 2 duplicates!
[Date] 'receipt_date' is read as text (str) -> needs conversion!
[Numeric] 'received_quantity' is read as text (str) -> needs cleaning!





In [17]:
# 1. Replace cells that contain only spaces (apparently empty) with NaN
dataframes["goods_receipts"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Replace specific keywords with real NaN
dataframes["goods_receipts"].replace(["N/A", "-", "Unknown", "null", ""], np.nan, inplace=True)

,receipt_number,purchase_order_number,purchase_order_line,receipt_date,receiving_facility_code,supplier_partner_code,received_quantity,received_unit,batch_reference,inspection_status,storage_location_code,receipt_status
0,RCV-2014-00001,PO-2014-00001,1,2014-07-08,FCL-1007,BP-30078,73,case,B-2014-44545,held,BIN-35,quality_hold
1,RCV-2015-00002,PO-2015-00001,1,2015-02-02,FCL-1010,BP-30068,0.9,service,B-2015-58495,accepted,BIN-36,posted
2,RCV-2015-00003,PO-2015-00002,1,2015-02-28,FCL-1030,BP-30066,1,service,B-2015-48474,held,BIN-17,quality_hold
3,RCV-2015-00004,PO-2015-00003,1,2015-03-02,FCL-1010,BP-30080,3,pack,B-2015-71913,accepted,BIN-16,posted
4,RCV-2015-00005,PO-2015-00004,1,2015-02-21,FCL-1022,BP-30021,1,grant,B-2015-31154,accepted,BIN-42,posted
...,...,...,...,...,...,...,...,...,...,...,...,...
607,RCV-2025-00608,PO-2025-00061,1,2025-06-28,FCL-1011,BP-30056,1,service,B-2025-42727,accepted,BIN-12,posted
608,RCV-2025-00609,PO-2025-00062,1,2025-06-30,FCL-1020,BP-30028,21,pallet,B-2025-90726,accepted,BIN-07,posted
609,RCV-2025-00610,PO-2025-00063,1,30/06/2025,FCL-1014,BP-30055,24,case,B-2025-90528,accepted,BIN-26,posted
610,RCV-2022-00319,PO-2022-00007,1,2022-02-22,FCL-1020,BP-30026,30,pallet,B-2022-05211,accepted,BIN-42,posted


In [18]:
# fix the date
dataframes["goods_receipts"]["receipt_date"] = parse_dates_robust(dataframes["goods_receipts"]["receipt_date"])

In [19]:
# remove duplicates
dataframes["goods_receipts"] = dataframes["goods_receipts"].drop_duplicates(subset=['receipt_number']).copy()

In [20]:
# fix the type
str_to_num("goods_receipts","received_quantity")

#### Cleaning of inventory_adjustments

Problems: date read as text, quantities stored as text with a unit. Fixes: conversion of the date and of the quantities **keeping the sign** (a negative adjustment is a stock withdrawal).

In [21]:
quick_review(dataframes["inventory_adjustments"], "inventory_adjustments.csv", data_dictionary)

====== Full review of inventory_adjustments.csv ======
Size: 140 rows, 11 columns
Duplicate rows: 0

[Gaps found] Missing values (NaN) per column:
approved_by_role             1
related_receipt_number     117
related_shipment_number    103
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'adjustment_number': unique and valid.
[Date] 'adjustment_date' is read as text (str) -> needs conversion!
[Numeric] 'adjustment_quantity' is read as text (str) -> needs cleaning!





In [22]:
# fix the date
dataframes["inventory_adjustments"]["adjustment_date"] = parse_dates_robust(dataframes["inventory_adjustments"]["adjustment_date"])

In [23]:
# fix the type
str_to_num("inventory_adjustments","adjustment_quantity")

In [24]:
# 1. Replace cells that contain only spaces (apparently empty) with NaN
dataframes["inventory_adjustments"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Replace specific keywords with real NaN
dataframes["inventory_adjustments"].replace(["N/A", "-", "Unknown", "null", ""], np.nan, inplace=True)

,adjustment_number,facility_code,adjustment_date,item_category,adjustment_quantity,unit,adjustment_reason,approved_by_role,related_receipt_number,related_shipment_number,adjustment_status
0,ADJ-2014-00001,FCL-1007,2014-07-18,medical_consumables,-1.58,case,specialist programme issue,clinical stores manager,NaN,NaN,posted
1,ADJ-2015-00002,FCL-1007,2015-09-10,medical_consumables,-2.70,case,specialist programme issue,specialist programme administrator,NaN,NaN,posted
2,ADJ-2015-00003,FCL-1024,2015-10-30,monitoring_equipment,3.68,kit,late receipt posting,warehouse manager,RCV-2022-00361,NaN,posted
3,ADJ-2016-00004,FCL-1007,2016-01-20,biological_sample,6.52,sealed_lot,late receipt posting,finance controller,NaN,NaN,pending_review
4,ADJ-2016-00005,FCL-1010,2016-02-18,medical_consumables,-2.43,case,procedure stock issue,quality manager,NaN,NaN,posted
...,...,...,...,...,...,...,...,...,...,...,...
135,ADJ-2025-00136,FCL-1017,2025-05-11,monitoring_equipment,1.00,kit,transfer posting correction,inventory supervisor,NaN,NaN,posted
136,ADJ-2025-00137,FCL-1030,2025-06-20,monitoring_equipment,1.00,kit,damaged packaging write-off,quality manager,NaN,NaN,approved
137,ADJ-2025-00138,FCL-1021,2025-06-20,monitoring_equipment,6.24,kit,transfer posting correction,finance controller,NaN,NaN,approved
138,ADJ-2025-00139,FCL-1015,2025-06-25,medical_consumables,-2.46,case,damaged packaging write-off,finance controller,NaN,NaN,approved


#### Cleaning of inventory_month_end

The key is composite (site + month + category): the "duplicates" reported column by column are normal. On the full key, 2 duplicate rows are removed. Months are converted to dates and quantities to numbers.

In [25]:
quick_review(dataframes["inventory_month_end"], "inventory_month_end.csv", data_dictionary)

====== Full review of inventory_month_end.csv ======
Size: 502 rows, 11 columns
Duplicate rows: 2

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'facility_code' contains 497 duplicates!
[Primary key] 'reporting_month' contains 482 duplicates!
[Primary key] 'item_category' contains 497 duplicates!
[Numeric] 'opening_stock' is read as text (str) -> needs cleaning!
[Numeric] 'received_quantity' is read as text (str) -> needs cleaning!
[Numeric] 'outgoing_quantity' is read as text (str) -> needs cleaning!
[Numeric] 'documented_internal_use': number format OK.
[Numeric] 'documented_waste' is read as text (str) -> needs cleaning!
[Numeric] 'closing_stock' is read as text (str) -> needs cleaning!





In [26]:
dataframes["inventory_month_end"]["reporting_month"] = parse_dates_robust(
    dataframes["inventory_month_end"]["reporting_month"]
)

In [27]:
keys = ['facility_code', 'reporting_month', 'item_category']
print("Duplicates on the composite key:", dataframes["inventory_month_end"].duplicated(subset=keys).sum())

df_month = dataframes["inventory_month_end"]

# Show all rows involved in the duplicates
conflicting_rows = df_month[df_month.duplicated(subset=keys, keep=False)]
display(conflicting_rows.sort_values(by=keys))



Duplicates on the composite key: 2


,facility_code,reporting_month,item_category,opening_stock,received_quantity,outgoing_quantity,documented_internal_use,documented_waste,closing_stock,unit,inventory_status
170,FCL-1019,2021-05-01,protective_equipment,57.10,11.93,0.00,1.03,2.10,64.71,case,closed
501,FCL-1019,2021-05-01,protective_equipment,57.10,11.93,0.00,1.03,2.10,64.71,case,closed
284,FCL-1020,2019-08-01,agricultural_product,43.82,10.93,0.00,3.98,0.94,51.14,pallet,closed
500,FCL-1020,2019-08-01,agricultural_product,43.82,10.93,0.00,3.98,0.94,51.14,pallet,closed


In [28]:
dataframes["inventory_month_end"] = df_month.drop_duplicates(subset=keys, keep='first').copy()

In [29]:
# fix the type
str_to_num("inventory_month_end","opening_stock")
str_to_num("inventory_month_end","received_quantity")
str_to_num("inventory_month_end","outgoing_quantity")
str_to_num("inventory_month_end","documented_waste")
str_to_num("inventory_month_end","closing_stock")

#### Cleaning of purchase_order_lines

Composite key (order number + line): 1 duplicate removed. Dates converted, quantities and prices converted to numbers (decimal commas, attached units).

In [30]:
quick_review(dataframes["purchase_order_lines"], "purchase_order_lines.csv", data_dictionary)

====== Full review of purchase_order_lines.csv ======
Size: 651 rows, 16 columns
Duplicate rows: 1

[Gaps found] Missing values (NaN) per column:
declared_cost_center    3
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'purchase_order_number' contains 1 duplicates!
[Primary key] 'purchase_order_line' contains 650 duplicates!
[Date] 'order_date' is read as text (str) -> needs conversion!
[Numeric] 'ordered_quantity' is read as text (str) -> needs cleaning!
[Numeric] 'unit_price' is read as text (str) -> needs cleaning!
[Date] 'requested_delivery_date' is read as text (str) -> needs conversion!





In [31]:
keys = ['purchase_order_number', 'purchase_order_line']
print("Duplicates on the composite key:", dataframes["purchase_order_lines"].duplicated(subset=keys).sum())

df_po = dataframes["purchase_order_lines"]

# Show all rows involved in the duplicates
conflicting_rows = df_po[df_po.duplicated(subset=keys, keep=False)]
display(conflicting_rows.sort_values(by=keys))

Duplicates on the composite key: 1


,purchase_order_number,purchase_order_line,order_date,ordering_facility_code,supplier_partner_code,item_code,item_description,item_category,ordered_quantity,ordered_unit,unit_price,currency,requested_delivery_date,authorization_class,declared_cost_center,order_status
348,PO-2022-00022,1,2022-02-09,FCL-1019,BP-30016,ITM-CSE-002,Insulated transport container,cold_storage_equipment,21,unit,3052.22,AVL,2022-03-17,ordinary,Facilities,completed
650,PO-2022-00022,1,2022-02-09,FCL-1019,BP-30016,ITM-CSE-002,Insulated transport container,cold_storage_equipment,21,unit,3052.22,AVL,2022-03-17,ordinary,Facilities,completed


In [32]:
dataframes["purchase_order_lines"] = df_po.drop_duplicates(subset=keys, keep='first').copy()

In [33]:
# fix the date
dataframes["purchase_order_lines"]["order_date"] = parse_dates_robust(dataframes["purchase_order_lines"]["order_date"])
dataframes["purchase_order_lines"]["requested_delivery_date"] = parse_dates_robust(dataframes["purchase_order_lines"]["requested_delivery_date"])

In [34]:
# fix the type
str_to_num("purchase_order_lines","ordered_quantity")
str_to_num("purchase_order_lines","unit_price")

In [35]:
# 1. Replace cells that contain only spaces (apparently empty) with NaN
dataframes["purchase_order_lines"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Replace specific keywords with real NaN
dataframes["purchase_order_lines"].replace(["N/A", "-", "Unknown", "null", ""], np.nan, inplace=True)

,purchase_order_number,purchase_order_line,order_date,ordering_facility_code,supplier_partner_code,item_code,item_description,item_category,ordered_quantity,ordered_unit,unit_price,currency,requested_delivery_date,authorization_class,declared_cost_center,order_status
0,PO-2014-00001,1,2014-06-24,FCL-1007,BP-30078,ITM-MED-001,General clinical consumable case,medical_consumables,73.0,case,157.85,VLN,2014-07-08,restricted,Clinical Programmes,completed
1,PO-2015-00001,1,2015-01-22,FCL-1010,BP-30068,ITM-TRN-001,Regional refrigerated transport service,transport_service,1.0,service,3482.27,VLN,2015-02-01,ordinary,Clinical Services,completed
2,PO-2015-00002,1,2015-01-29,FCL-1030,BP-30066,ITM-COM-001,Secure communication service,communication_service,1.0,service,1292.19,KRN,2015-02-26,ordinary,General Operations,completed
3,PO-2015-00003,1,2015-02-10,FCL-1010,BP-30080,ITM-LAB-002,Sample preparation material pack,laboratory_material,3.0,pack,458.67,VLN,2015-03-02,expedited,Clinical Services,completed
4,PO-2015-00004,1,2015-02-12,FCL-1022,BP-30021,ITM-GRA-001,Programme grant transfer,grant,1.0,grant,26933.28,KRN,2015-02-21,ordinary,Facilities,completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,PO-2025-00059,1,2025-05-22,FCL-1025,BP-30007,ITM-CSE-002,Insulated transport container,cold_storage_equipment,15.0,unit,3353.95,NDH,2025-06-20,ordinary,General Operations,completed
646,PO-2025-00060,1,2025-05-25,FCL-1019,BP-30078,ITM-PRO-002,Cold-handling protective supplies,protective_equipment,59.0,case,631.64,AVL,2025-06-07,expedited,Field Operations,completed
647,PO-2025-00061,1,2025-05-30,FCL-1011,BP-30056,ITM-TRN-001,Regional refrigerated transport service,transport_service,1.0,service,3681.92,VLN,2025-06-25,ordinary,Agricultural Distribution,completed
648,PO-2025-00062,1,2025-06-18,FCL-1020,BP-30028,ITM-AGR-001,Durian produce pallet,agricultural_product,21.0,pallet,1772.80,TMB,2025-07-02,ordinary,Facilities,completed


#### Cleaning of shipments

Composite key (shipment number + leg): 2 duplicates removed. Dispatch and arrival dates converted (several formats); the 17 missing arrival dates are left empty.

In [36]:
quick_review(dataframes["shipments"], "shipments.csv", data_dictionary)

====== Full review of shipments.csv ======
Size: 452 rows, 16 columns
Duplicate rows: 2

[Gaps found] Missing values (NaN) per column:
arrival_time    17
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'shipment_number' contains 2 duplicates!
[Primary key] 'shipment_leg' contains 451 duplicates!
[Date] 'dispatch_time' is read as text (str) -> needs conversion!
[Date] 'arrival_time' is read as text (str) -> needs conversion!
[Numeric] 'declared_quantity': number format OK.





In [37]:
keys = ['shipment_number', 'shipment_leg']
print("Duplicates on the composite key:", dataframes["shipments"].duplicated(subset=keys).sum())

df_exp = dataframes["shipments"]

# Show all rows involved in the duplicates
conflicting_rows = df_exp[df_exp.duplicated(subset=keys, keep=False)]
display(conflicting_rows.sort_values(by=keys))

Duplicates on the composite key: 2


,shipment_number,shipment_leg,dispatch_time,arrival_time,sender_partner_code,receiver_partner_code,origin_facility_code,destination_facility_code,carrier_partner_code,item_category,declared_quantity,declared_unit,temperature_control_class,authorization_class,declared_purpose,shipment_status
26,SHP-2016-00009,1,2016-09-05T13:00:00,2016-09-08T02:00:00,BP-30043,BP-30036,FCL-1010,FCL-1030,BP-30020,consulting_service,1,service,ambient,emergency,scheduled stock transfer,delivered
451,SHP-2016-00009,1,2016-09-05T13:00:00,2016-09-08T02:00:00,BP-30043,BP-30036,FCL-1010,FCL-1030,BP-30020,consulting_service,1,service,ambient,emergency,scheduled stock transfer,delivered
251,SHP-2022-00017,1,2022-02-16T14:00:00,2022-02-18T15:00:00,BP-30073,BP-30078,FCL-1019,FCL-1011,BP-30031,protective_equipment,3,case,ambient,ordinary,agricultural distribution,delivered
450,SHP-2022-00017,1,2022-02-16T14:00:00,2022-02-18T15:00:00,BP-30073,BP-30078,FCL-1019,FCL-1011,BP-30031,protective_equipment,3,case,ambient,ordinary,agricultural distribution,delivered


In [38]:
dataframes["shipments"] = df_exp.drop_duplicates(subset=keys, keep='first').copy()

In [39]:
# fix the date
dataframes["shipments"]["dispatch_time"] = parse_dates_robust(dataframes["shipments"]["dispatch_time"])
dataframes["shipments"]["arrival_time"] = parse_dates_robust(dataframes["shipments"]["arrival_time"])

#### Harmonisation of texts and categories (all tables)

Some categorical columns contain the same category written in several ways (`posted` / `POSTED`, `2-8C` / `2-8c`, `transfer posting correction` / `TRANSFER POSTING CORRECTION`) or with extra spaces (`"  scheduled  stock  transfer "`). Without correction, these variants are counted as different categories: they become artificially "rare" categories for one-hot encoding, clustering or Apriori.

We therefore remove superfluous spaces in all text columns, then, for each categorical column, replace each variant with the most frequent spelling of the category. Identifiers (codes) and proper names keep their case.

In [40]:
categorical_columns = {
    "goods_receipts": ["received_unit", "inspection_status", "receipt_status"],
    "inventory_adjustments": ["item_category", "unit", "adjustment_reason", "approved_by_role", "adjustment_status"],
    "inventory_month_end": ["item_category", "unit", "inventory_status"],
    "purchase_order_lines": ["item_category", "ordered_unit", "currency", "authorization_class",
                             "declared_cost_center", "order_status"],
    "shipments": ["item_category", "declared_unit", "temperature_control_class", "authorization_class",
                  "declared_purpose", "shipment_status"],
    "erp_facilities": ["facility_category", "region", "facility_status"],
    "erp_business_partners": ["partner_role", "billing_region", "shipping_region", "default_currency",
                              "payment_terms", "partner_status", "risk_class"],
}

for table_name, columns in categorical_columns.items():
    df = dataframes[table_name]
    before = {col: df[col].nunique() for col in columns}
    normalize_text(df, columns)
    for col in columns:
        after = df[col].nunique()
        if after != before[col]:
            print(f"{table_name}.{col}: {before[col]} -> {after} categories")

goods_receipts.receipt_status: 3 -> 2 categories
inventory_adjustments.adjustment_reason: 12 -> 10 categories
inventory_month_end.inventory_status: 5 -> 3 categories


purchase_order_lines.item_category: 23 -> 18 categories
shipments.temperature_control_class: 6 -> 3 categories
shipments.declared_purpose: 12 -> 8 categories
shipments.shipment_status: 7 -> 4 categories
erp_business_partners.partner_role: 6 -> 5 categories


In [41]:
# 1. Output folder
output_dir = Path("../data/logistics/clean")
output_dir.mkdir(parents=True, exist_ok=True)
counter = 0

# 2. The 7 logistics tables to export
logistics_tables = [
    "goods_receipts",
    "inventory_adjustments",
    "inventory_month_end",
    "purchase_order_lines",
    "shipments",
    "erp_facilities",
    "erp_business_partners"
]

for table_name in logistics_tables:
    if table_name in dataframes:
        full_path = output_dir / f"{table_name}_clean.csv"
        dataframes[table_name].to_csv(full_path, index=False, na_rep="NaN")
        counter += 1
        print(f"Exported: {table_name}_clean.csv")

print(f"\n{counter} files exported to: {output_dir}")

Exported: goods_receipts_clean.csv
Exported: inventory_adjustments_clean.csv
Exported: inventory_month_end_clean.csv
Exported: purchase_order_lines_clean.csv
Exported: shipments_clean.csv
Exported: erp_facilities_clean.csv
Exported: erp_business_partners_clean.csv

7 files exported to: ..\data\logistics\clean


In [42]:
logistics_tables = [
    "goods_receipts",
    "inventory_adjustments",
    "inventory_month_end",
    "purchase_order_lines",
    "shipments",
    "erp_facilities",
    "erp_business_partners"
]

for table_name in logistics_tables:
    quick_review(dataframes[table_name], table_name +".csv", data_dictionary)

====== Full review of goods_receipts.csv ======
Size: 610 rows, 12 columns
Duplicate rows: 0

[Gaps found] Missing values (NaN) per column:
inspection_status    4
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'receipt_number': unique and valid.
[Date] 'receipt_date': date format OK.
[Numeric] 'received_quantity': number format OK.



====== Full review of inventory_adjustments.csv ======
Size: 140 rows, 11 columns
Duplicate rows: 0

[Gaps found] Missing values (NaN) per column:
approved_by_role             1
related_receipt_number     117
related_shipment_number    103
dtype: int64

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key] 'adjustment_number': unique and valid.
[Date] 'adjustment_date': date format OK.
[Numeric] 'adjustment_quantity': number format OK.



====== Full review of inventory_month_end.csv ======
Size: 500 rows, 11 columns
Duplicate rows: 0

--- Check by logical type (dictionary vs DataFrame) ---
[Primary key